# TPC-H Benchmark Tutorial (RumbleDB)
<span style="font-size: 14px; color: #555;">A step-by-step notebook to load TPC-H data, create collections in Hive/Delta/Iceberg, and benchmark Query 1.</span>



## Step 1: Launch RumbleDB
Configure a single session with both Delta and Iceberg enabled. Use a catalog name for Iceberg and set driver memory high enough for your scale factors.

**Edit before running:**
- `iceberg_catalog`
- `spark.driver.memory`



In [ ]:
from jsoniq import RumbleSession

iceberg_catalog = 'iceberg'

# Change the driver memory accordingly
rumble = RumbleSession.builder.withDelta().withIceberg([iceberg_catalog]) \
    .config("spark.driver.host", "localhost") \
    .config("spark.driver.memory", "20g") \
    .getOrCreate()

%load_ext jsoniqmagic

## Step 2: Prepare TPC-H Data Files
This section expects `dbgen` output in local folders for SF1, SF10, and SF30. Keep the directory layout consistent so the notebook can find the `.tbl` files.

If your `dbgen` output lives elsewhere, update the paths in the next cell.




In [ ]:
# Get TPC-H .tbl files - make sure the relative path is correct (path to generated tables)
files_sf1 = !ls ./TPC-H-V3.0.1/dbgen/sf1/*.tbl
files_sf10 = !ls ./TPC-H-V3.0.1/dbgen/sf10/*.tbl
files_sf30 = !ls ./TPC-H-V3.0.1/dbgen/sf30/*.tbl

files_sf1 = tuple(files_sf1)
files_sf10 = tuple(files_sf10)
files_sf30 = tuple(files_sf30)

# Setup TPC-H table names
tables = ('customer', 'lineitem', 'nation', 'orders', 'part', 'partsupp', 'region', 'supplier')

# Setup TPC-H table columns
table_columns = {
    'customer': [
        "c_custkey", "c_name", "c_address", "c_nationkey",
        "c_phone", "c_acctbal", "c_mktsegment", "c_comment"
    ],
    'lineitem': [
        "l_orderkey", "l_partkey", "l_suppkey", "l_linenumber",
        "l_quantity", "l_extendedprice", "l_discount", "l_tax",
        "l_returnflag", "l_linestatus", "l_shipdate",
        "l_commitdate", "l_receiptdate", "l_shipinstruct",
        "l_shipmode", "l_comment"
    ],
    'nation': [
        "n_nationkey", "n_name", "n_regionkey", "n_comment"
    ],
    'orders': [
        "o_orderkey", "o_custkey", "o_orderstatus", "o_totalprice",
        "o_orderdate", "o_orderpriority", "o_clerk", "o_shippriority", "o_comment"
    ],
    'part': [
        "p_partkey", "p_name", "p_mfgr", "p_brand",
        "p_type", "p_size", "p_container", "p_retailprice", "p_comment"
    ],
    'partsupp': [
        "p_partkey", "p_suppkey", "p_availqty",
        "p_supplycost", "p_comment"
    ],
    'region': [
        "r_regionkey", "r_name", "r_comment"
    ],
    'supplier': [
        "s_suppkey", "s_name", "s_address", "s_nationkey",
        "s_phone", "s_acctbal", "s_comment"
    ]
}

# Setup TPC-H table column types
table_types = {
    'customer': [
        "int", "string", "string", "int",
        "string", "decimal", "string", "string"
    ],
    'lineitem': [
        "int", "int", "int", "int",
        "decimal", "decimal", "decimal", "decimal",
        "string", "string", "date",
        "date", "date", "string",
        "string", "string"
    ],  
    'nation': [
        "int", "string", "int", "string"
    ],
    'orders': [
        "int", "int", "string", "decimal",
        "date", "string", "string", "int", "string"
    ],
    'part': [
        "int", "string", "string", "string",
        "string", "int", "string", "decimal", "string"
    ],
    'partsupp': [
        "int", "int", "int",
        "decimal", "string"
    ],
    'region': [
        "int", "string", "string"
    ],
    'supplier': [
        "int", "string", "string", "int",
        "string", "decimal", "string"
    ]
}


# Bind variables for use in JSONiq cells
rumble.bind("$files_sf1", files_sf1)
rumble.bind("$files_sf10", files_sf10)
rumble.bind("$files_sf30", files_sf30)

rumble.bind("$tables", tables)
rumble.bind("$table_columns", table_columns)
rumble.bind("$table_types", table_types)

# Setup Delta
import os
delta_dir = "./delta-tpch"
os.makedirs(delta_dir, exist_ok=True)
rumble.bind("$delta_dir", delta_dir)

# Debug output
print("File paths:", files_sf1, files_sf10, files_sf30)
print("Table names:", tables)
print("Table columns:", table_columns)
print("Table types:", table_types)

## Step 3: Create Collections (Hive, Delta, Iceberg)
The next cells define timing helpers and generate typed JSONiq queries that create collections from the raw `.tbl` files.

**Outputs:**
- `creation_benchmark_sf1.csv`
- `creation_benchmark_sf10.csv`
- `creation_benchmark_sf30.csv`



In [ ]:
import time, math, pandas as pd

# --- Benchmark helpers ---
def _stats(times):
    n = len(times)
    mean = sum(times) / n if n else float("nan")
    std = math.sqrt(sum((t - mean) ** 2 for t in times) / (n - 1)) if n >= 2 else float("nan")
    median = sorted(times)[n // 2] if n else float("nan")
    return mean, std, median

def measure_collection_creation(label, create_code, delete_code, runs=5, warmup=1):
    # Warmup runs (not recorded)
    for _ in range(warmup):
        rumble.jsoniq(create_code).applyPUL()
        rumble.jsoniq(delete_code).applyPUL()

    # Timed runs
    times = []
    for i in range(runs):
        t0 = time.perf_counter()
        rumble.jsoniq(create_code).applyPUL()
        times.append(time.perf_counter() - t0)

        if i < runs - 1:
            rumble.jsoniq(delete_code).applyPUL()

    mean, std, median = _stats(times)
    return {
        "label": label,
        "runs": runs,
        "warmup": warmup,
        "mean": mean,
        "std": std,
        "median": median,
        "times": times,
    }


### Step 3.1: Build Typed JSONiq Create Statements
We generate a single JSONiq query that maps each TPC-H table to a typed object constructor.



In [ ]:
# --- Generate typed JSONiq for all tables (single query) ---
def _type_decl(table, columns, types):
    fields = [f"\"{c}\": \"{t}\"" for c, t in zip(columns, types)]
    return "declare type local:{}_t as {{ {} }};".format(table, ", ".join(fields))

def _obj_constructor(columns, types):
    parts = []
    for idx, (col, typ) in enumerate(zip(columns, types), start=1):
        if typ == "int":
            expr = f"xs:int($col[{idx}])"
        elif typ == "decimal":
            expr = f"xs:decimal($col[{idx}])"
        elif typ == "date":
            expr = f"xs:date($col[{idx}])"
        else:
            expr = f"$col[{idx}]"
        parts.append(f"\"{col}\": {expr}")
    return "{ " + ", ".join(parts) + " }"

def _table_case(table):
    obj = _obj_constructor(table_columns[table], table_types[table])
    return (
        f"\n  else if ($table eq \"{table}\") then"
        f"\n    create collection {{TARGET}} with"
        f"\n      validate type local:{table}_t* {{"
        f"\n        for $line in text-file($file)"
        f"\n        let $col := tokenize($line, \"\\\\|\")"
        f"\n        return {obj}"
        f"\n      }}"
    )

def build_typed_create_query(target_expr, files_var):
    decls = ["declare namespace local = \"local\";"]
    decls += [_type_decl(t, table_columns[t], table_types[t]) for t in tables]

    first = tables[0]
    obj0 = _obj_constructor(table_columns[first], table_types[first])
    cases = [
        f"\nif ($table eq \"{first}\") then"
        f"\n  create collection {{TARGET}} with"
        f"\n    validate type local:{first}_t* {{"
        f"\n      for $line in text-file($file)"
        f"\n      let $col := tokenize($line, \"\\\\|\")"
        f"\n      return {obj0}"
        f"\n    }}"
    ]
    for t in tables[1:]:
        cases.append(_table_case(t))
    cases.append("\n  else ()\n")

    query = ("\n".join(decls)
        + f"\nfor $table in $tables\ncount $i\nlet $file := {files_var}[$i]\nreturn\n"
        + "".join(cases)
    )
    return query.replace("{TARGET}", target_expr)


### Step 3.2: Assemble Create/Delete Queries by Scale Factor
This cell builds the concrete create/delete JSONiq for each storage target and scale factor. You can adjust the output paths or naming here if needed.



In [ ]:
# --- Build typed create/delete queries for each SF + target ---
delete_template = r"""
for $table in $tables
return delete collection {TARGET}
"""

sfs = [
    ("sf1", "$files_sf1"),
    ("sf10", "$files_sf10"),
    ("sf30", "$files_sf30"),
]

delta_dir_base = "./delta-tpch"

create_delete_by_sf = []
for sf, files_var in sfs:
    delta_dir_sf = f"{delta_dir_base}/{sf}"
    os.makedirs(delta_dir_sf, exist_ok=True)

    # construct collection expressions for each target
    hive_collection = f"table(concat('{sf}_', $table))"
    delta_collection = f"delta-file(concat('{delta_dir_sf}/', $table))"
    iceberg_collection = f"iceberg-table(concat('{iceberg_catalog}.{sf}_', $table))"
    
    # construct create queries for each target
    hive_create = build_typed_create_query(hive_collection, files_var)
    delta_create = build_typed_create_query(delta_collection, files_var)
    iceberg_create = build_typed_create_query(iceberg_collection, files_var)

    # construct delete queries for each target
    hive_delete = delete_template.replace("{TARGET}", hive_collection)
    delta_delete = delete_template.replace("{TARGET}", delta_collection)
    iceberg_delete = delete_template.replace("{TARGET}", iceberg_collection)

    # store the create/delete queries for this SF
    create_delete_by_sf.append((sf, {
        "hive": (hive_create, hive_delete),
        "delta": (delta_create, delta_delete),
        "iceberg": (iceberg_create, iceberg_delete),
    }))


### Step 3.3: Run Creation Benchmarks
Runs the timed creation loop for SF1, SF10, and SF30 and writes a CSV per scale factor.



In [ ]:
# --- Run creation benchmark for all SFs ---
create_res  = [[], [], []] # one for each sf 
for sf_idx, (sf, targets) in enumerate(create_delete_by_sf):
    print(f"Running creation benchmark for SF={sf}...")
    for label, (create_code, delete_code) in targets.items():
        print(f"Measuring {label}...")
        res = measure_collection_creation(f"{label}_{sf}", create_code, delete_code)
        create_res[sf_idx].append(res)

    df = pd.DataFrame(create_res[sf_idx])[["label","runs","warmup","mean","std","median"]]

    print(f"Driver memory: {rumble.sparkContext.getConf().get("spark.driver.memory")}\n")
    print(df)

    csv_path = f"creation_benchmark_{sf}.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nSaved to {csv_path}\n")

print("Creation benchmark completed for all SFs.")

## Step 4: Query Definitions and Benchmarks
Below we define the helper that measures query execution time and then run TPC-H Query 1 across all formats and scale factors.



In [ ]:
def measure_query(label, query, runs=5, warmup=1):
    # warmup
    for _ in range(warmup):
        rumble.jsoniq(query).df().count()

    # timed runs
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        rumble.jsoniq(query).df().count()
        times.append(time.perf_counter() - t0)

    mean, std, median = _stats(times)
    return {
        "label": label,
        "runs": runs,
        "warmup": warmup,
        "mean": mean,
        "median": median,
        "std": std,
    }

### Step 4.1: Q1 Template
This is the full TPC-H Query 1 expressed in JSONiq. The `{SOURCE}` placeholder is replaced with the target collection (Hive/Delta/Iceberg).


**Outputs:**
- `q1_benchmark_sf1.csv`
- `q1_benchmark_sf10.csv`
- `q1_benchmark_sf30.csv`


In [ ]:
q1_template = r"""
declare namespace local = "local";

declare type local:q1_t as {
  "l_returnflag": "string",
  "l_linestatus": "string",
  "sum_qty": "decimal",
  "sum_base_price": "decimal",
  "sum_disc_price": "decimal",
  "sum_charge": "decimal",
  "avg_qty": "decimal",
  "avg_price": "decimal",
  "avg_disc": "decimal",
  "count_order": "integer"
};

let $delta-days := 90
let $cutoff-date :=
  xs:date("1998-12-01")
  - xs:dayTimeDuration(concat("P", string($delta-days), "D"))

return
  validate type local:q1_t* {
    for $l in {SOURCE}

    where $l.l_shipdate le $cutoff-date

    let $quantity      := xs:decimal($l.l_quantity)
    let $extendedprice := xs:decimal($l.l_extendedprice)
    let $discount      := xs:decimal($l.l_discount)
    let $tax           := xs:decimal($l.l_tax)

    group by
      $returnflag := $l.l_returnflag,
      $linestatus := $l.l_linestatus

    let $sum_qty        := sum($quantity)
    let $sum_base_price := sum($extendedprice)
    let $sum_disc_price := sum(
      for $i in count($extendedprice)
      return $extendedprice[$i] * (1 - $discount[$i])
    )
    let $sum_charge     := sum(
      for $i in count($extendedprice)
      return $extendedprice[$i] * (1 - $discount[$i]) * (1 + $tax[$i])
    )
    let $avg_qty        := avg($quantity)
    let $avg_price      := avg($extendedprice)
    let $avg_disc       := avg($discount)
    let $count_order    := count($quantity)

    order by
      $returnflag ascending,
      $linestatus ascending

    return {
      "l_returnflag"  : $returnflag,
      "l_linestatus"  : $linestatus,
      "sum_qty"       : $sum_qty,
      "sum_base_price": $sum_base_price,
      "sum_disc_price": $sum_disc_price,
      "sum_charge"    : $sum_charge,
      "avg_qty"       : $avg_qty,
      "avg_price"     : $avg_price,
      "avg_disc"      : $avg_disc,
      "count_order"   : $count_order
    }
  }
"""

# --- Q1 benchmark for all SFs ---
q1_res = []
for sf, _files_var in sfs:
    hive_collection = f"table('{sf}_lineitem')"
    delta_collection = f"delta-file(concat('{delta_dir_base}/{sf}/', 'lineitem'))"
    iceberg_collection = f"iceberg-table('{iceberg_catalog}.{sf}_lineitem')"

    hive_query = q1_template.replace("{SOURCE}", hive_collection)
    delta_query = q1_template.replace("{SOURCE}", delta_collection)
    iceberg_query = q1_template.replace("{SOURCE}", iceberg_collection)

    print(f"Running Q1 benchmark for SF={sf}...")
    print("Measuring Hive query...")
    q1_res.append(measure_query(f"hive_{sf}", hive_query))

    print("Measuring Delta query...")
    q1_res.append(measure_query(f"delta_{sf}", delta_query))
    
    print("Measuring Iceberg query...")
    q1_res.append(measure_query(f"iceberg_{sf}", iceberg_query))

    df = pd.DataFrame(q1_res)[["label","runs","warmup","mean","std","median"]]
    print(f"\nDriver memory: {rumble.sparkContext.getConf().get('spark.driver.memory')}\n")
    print(df)

    csv_path = f"q1_benchmark_{sf}.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nSaved to {csv_path}\n")